# Analyse the Results of Running Moran Process Experiment on Different Graphs
This is the newest version of this analysis file, where I can merge the csv of different jobs. 

## Setup

Imports, configuration, and data loading. Set `BATCH_NAME` below to point at whichever batch you want to analyze.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process 

import sys
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
# import numpy as np
import seaborn as sns
import os
from pathlib import Path
from moran_process.analysis.analysis_utils import *
# change this if on a different computer!
from moran_process.core.population_graph import GRAPH_PROPS
# Set aesthetic parameters for "publication-quality" plots
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['lines.linewidth'] = 2.5


In [ ]:

BATCH_NAME = '2026-05-20_scaling_study_3'  # <-- change this to whatever batch you want to analyse
CATEGORY_FILTER = 'Random'  # set to e.g. 'Random', 'Complete', 'Cycle' to filter; None = all categories
ANIMAL_CATEGORIES = ["Mammalian", "Fish", "Avian"]



In [ ]:
ROOT = Path(os.getcwd()) 

# Now define your paths relative to ROOT
BATCH_DIR = ROOT / "simulation_data" / BATCH_NAME
FIGURES_DIR = BATCH_DIR / "figures"


In [ ]:
# --- Batch provenance ---
batch_info = load_batch_info(BATCH_DIR)
BATCH_LABEL = batch_info['name']
plot_batch_info_card(batch_info, figures_dir=FIGURES_DIR)

In [ ]:
import glob

output_file = os.path.join(BATCH_DIR, f"temp_raw_results.csv") 
tmp_results_path = os.path.join(BATCH_DIR, "tmp", "results")
all_files = glob.glob(os.path.join(tmp_results_path, "raw_results_job_*.csv"))
print(f"Found {len(all_files)} files in temp results directory: {tmp_results_path}.")


### The Graph Features we analyse: 


In [ ]:
GRAPH_PROPERTY_DESCRIPTION.keys()

In [ ]:
GRAPH_PROPERTY_DESCRIPTION
for prop, desc in GRAPH_PROPERTY_DESCRIPTION.items():
    print(f"{prop.title().replace('_', ' '):40}: {desc}")

In [ ]:
results_df_path = aggregate_results_no_load(batch_dir=BATCH_DIR, delete_temp=False)

In [ ]:

lazy_df = pl.scan_csv(results_df_path)
schema = lazy_df.collect_schema()
n_rows = lazy_df.select(pl.len()).collect()[0, 0]
print("columns:", list(schema.keys()))
print(f"Raw data: {n_rows:,} rows x {len(schema)} columns")


In [ ]:
# --- Aggregating the Raw Simulations Result ---

df_graphs = pd.read_csv(os.path.join(BATCH_DIR, 'graph_props.csv'))
graph_statistics_path = BATCH_DIR / "graph_statistics.csv"
if not graph_statistics_path.exists:

    agg_results_df = (
        lazy_df
        .with_columns(
            pl.when(pl.col('fixation')).then(pl.col('steps')).otherwise(None).alias('steps_success')
        )
        .group_by(['wl_hash', 'r', 'graph_name'])
        .agg([
            pl.col('fixation').mean().alias('prob_fixation'),
            pl.col('steps_success').median().alias('median_steps'),
            pl.col('steps_success').mean().alias('mean_steps'),
            pl.col('steps_success').std().alias('std_steps'),
            pl.col('steps_success').quantile(0.25).alias('q25_steps'),
            pl.col('steps_success').quantile(0.75).alias('q75_steps'),
            (pl.col('steps_success').quantile(0.75) - pl.col('steps_success').quantile(0.25)).alias('iqr_steps'),
            pl.col('fixation').count().alias('n_grouped'),
        ])
        .collect(engine='streaming')  # constant-memory aggregation, safe for large batches
        .to_pandas()
    )

    print("Shape before merging: ", agg_results_df.shape)

    analysis_df = pd.merge(
        agg_results_df,
        df_graphs,
        on=['wl_hash', 'graph_name'],
        how='left',
        suffixes=('', '_db')
    )
    analysis_df['z_order'] = (analysis_df['category'] != 'Random').astype(int)
    analysis_df = analysis_df.sort_values('z_order').drop(columns='z_order')
    analysis_df.to_csv(os.path.join(BATCH_DIR, 'graph_statistics.csv'), index=False)
else: 
    print(f"Aggregated Simulation Graph Statistics already exists! Loading {graph_statistics_path}...")
    analysis_df = pd.read_csv(graph_statistics_path)
print("Shape after merging: ", analysis_df.shape)

if CATEGORY_FILTER is not None:
    analysis_df = analysis_df[analysis_df['category'] == CATEGORY_FILTER].copy()
    print(f"Filtered to '{CATEGORY_FILTER}': {len(analysis_df):,} rows")

print(f"Graph aggregated simuation statistics table columns: {analysis_df.columns}")
analysis_df.tail(20)


In [ ]:
plot_steps_histogram(
    analysis_df,
    metric='mean_steps',
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_steps_histogram(
    analysis_df,
    metric='mean_steps',
    category='Random',
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
categories = sorted(analysis_df['category'].dropna().unique().tolist())
category_color_dict = generate_robust_color_dict(analysis_df, CATEGORY_COLOR_DICT)

print("Color dictionary:")
for cat, color in category_color_dict.items():
    print(f"  {cat:15}: {color}")

In [ ]:
plot_steps_violin(
    results_csv_path=results_df_path,
    df_graphs=df_graphs,
    color_dict=category_color_dict,
    categories=categories,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_two_property_effect(
    df=analysis_df,
    x_prop='n_nodes',
    y_prop='n_edges',
    outcome='mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_two_property_effect_hexbin(
    df=analysis_df,
    x_prop='n_nodes',
    y_prop='n_edges',
    outcome='mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_two_property_effect_hexbin(
    df=analysis_df,
    x_prop='density',
    y_prop='n_nodes',
    outcome='mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_outcome_vs_property(
    analysis_df,
    'density',
    'mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_outcome_vs_property(
    analysis_df,
    'transitivity',
    'mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)
plot_two_property_effect_hexbin(
    df=analysis_df,
    x_prop='transitivity',
    y_prop='n_nodes',
    outcome='mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_outcome_vs_property(
    analysis_df,
    'max_degree',
    'prob_fixation',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
df_to_plot = analysis_df[analysis_df['r'] == 1.1]

plot_outcome_vs_property(
    analysis_df,
    'prob_fixation',
    'mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)
plot_outcome_vs_property(
    analysis_df,
    'mean_steps',
    'prob_fixation',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

## Per-Property Effect on Fixation Time (`r = 1.1`)

Each plot below sweeps one graph structural property against mean fixation time under weak selection (`r = 1.1`). Animal graphs are highlighted; random graphs form the background distribution.  Look for properties where the animal graphs sit outside the random cloud -- those are structural features that may explain amplifier or suppressor behavior.

In [ ]:
for prop in GRAPH_PROPS:
    plot_outcome_vs_property(
        df_to_plot, prop, 'mean_steps',
        density_threshold=50,
        color_dict=category_color_dict,
        highlight_categories=ANIMAL_CATEGORIES,
        figures_dir=FIGURES_DIR,
        batch_name=BATCH_LABEL,
    )

## Per-Property Effect on Fixation Probability (`r = 1.1`)

Same sweep as above but with fixation probability on the y-axis. Cross-reference with the fixation time plots: a true amplifier should show both elevated probability *and* shorter fixation time relative to the neutral drift baseline (`rho = 1/N`).

In [ ]:
for prop in GRAPH_PROPS:
    plot_outcome_vs_property(
        df_to_plot, prop, 'prob_fixation',
        density_threshold=50,
        color_dict=category_color_dict,
        highlight_categories=ANIMAL_CATEGORIES,
        figures_dir=FIGURES_DIR,
        batch_name=BATCH_LABEL,
    )

In [ ]:
plot_outcome_vs_property(
    df_to_plot,
    'degree_std',
    'mean_steps',
    density_threshold=50,
    color_dict=category_color_dict,
    batch_name=BATCH_LABEL,
)

plt.figure(figsize=(10, 8))
plt.hexbin(df_to_plot['degree_std'], df_to_plot['mean_steps'], gridsize=20, cmap='YlOrRd', mincnt=1)
plt.xlabel('degree_std')
plt.ylabel('mean_steps')
plt.colorbar(label='count')
plt.title('Hexbin plot: degree_std vs mean_steps')
plt.show()